# ViQAG Pipeline Training on Google Colab
## Vietnamese Question Answer Generation with AE + QG Fine-tuning

This notebook trains both Answer Extraction (AE) and Question Generation (QG) models using the ViQAG pipeline on AI Flashcard dataset.

## Step 1: Setup & Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch torchvision torchaudio
!pip install -q transformers datasets
!pip install -q underthesea
!pip install -q tqdm
!pip install -q fire
!pip install -q nltk
!pip install -q numpy pandas scikit-learn

## Step 2: Mount Google Drive & Upload Dataset

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create working directory
os.makedirs('/content/ViQAG_work', exist_ok=True)
os.chdir('/content/ViQAG_work')

print("✓ Google Drive mounted at /content/drive")
print("✓ Working directory: /content/ViQAG_work")

## Step 3: Copy Dataset from Drive or Upload Manually

In [ ]:
import shutil

# Option A: If you have the data in Google Drive
# Uncomment and modify the path below:
# shutil.copytree('/content/drive/MyDrive/ViQAG/data/examples_ai_flashcard', '/content/ViQAG_work/data/examples_ai_flashcard')

# Option B: Create data directory structure
os.makedirs('/content/ViQAG_work/data/examples_ai_flashcard', exist_ok=True)
os.makedirs('/content/ViQAG_work/data/processed_ai_flashcard', exist_ok=True)
os.makedirs('/content/ViQAG_work/plms', exist_ok=True)

print("✓ Data directories created")
print("\nMANUAL STEP: Upload these 3 files to /content/ViQAG_work/data/examples_ai_flashcard/")
print("  - train.jsonl (84 samples)")
print("  - validation.jsonl (30 samples)")
print("  - test.jsonl (30 samples)")
print("\nAfter upload, run the next cell.")

In [ ]:
# Verify dataset files exist
import os

data_dir = '/content/ViQAG_work/data/examples_ai_flashcard'
files = ['train.jsonl', 'validation.jsonl', 'test.jsonl']

for f in files:
    path = os.path.join(data_dir, f)
    if os.path.exists(path):
        with open(path, 'r') as file:
            lines = len(file.readlines())
        print(f"✓ {f}: {lines} samples")
    else:
        print(f"✗ {f}: NOT FOUND - Please upload it")

## Step 4: Data Preprocessing with qg_data.py

In [ ]:
# Create qg_data.py preprocessing module using Underthesea
import json
import os
import re
from underthesea import sent_tokenize
from tqdm import tqdm
from typing import Dict

HIGHLIGHT_TOKEN = '<hl>'

class QGDataProcessor:
    """Vietnamese QA preprocessing with highlight token representations"""
    
    def __init__(self):
        self.input_dir: str = 'data/examples'
        self.output_dir: str = 'data/processed_data'

    def get_sentence(self, document: str):
        """Tokenize Vietnamese text into sentences using Underthesea"""
        try:
            sentences = sent_tokenize(document)
            return [s.strip() for s in sentences if s.strip()]
        except:
            return [document]

    def jsonline_reader(self, filename: str):
        """Read JSONL with robust error handling for malformed JSON"""
        examples = []
        with open(filename, 'r', encoding='utf-8') as f_reader:
            for line_num, line in enumerate(f_reader, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    example = json.loads(line)
                    examples.append(example)
                except json.JSONDecodeError as e:
                    print(f"  ⚠ Line {line_num}: {str(e)[:50]}")
                    continue
        return examples

    def process_single_data(self, data: Dict):
        """Convert single raw json data into QG format with highlights"""
        try:
            example = {
                'question': data.get("question", "").strip(),
                'paragraph': data.get("context", "").strip(),
                'answer': data.get("answer", "").strip()
            }

            # Validate all required fields exist
            if not all([example['question'], example['paragraph'], example['answer']]):
                return None

            # Find answer position - skip if not found
            position = example['paragraph'].find(example['answer'])
            if position == -1:
                return None
            
            # Get sentences before answer
            before_tmp = self.get_sentence(example['paragraph'][:position])
            if len(before_tmp) == 0:
                before = ''
                before_sentence = ''
            else:
                if before_tmp[-1].endswith('.'):
                    before = ' '.join(before_tmp)
                    before_sentence = ''
                else:
                    before = ' '.join(before_tmp[:-1])
                    before_sentence = before_tmp[-1]
                    before_sentence = before_sentence if before_sentence.endswith(' ') else '{} '.format(before_sentence)
            
            # Get sentences after answer
            after_tmp = self.get_sentence(example['paragraph'][position + len(example['answer']):])
            if len(after_tmp) == 0:
                after = ''
                after_sentence = ''
            else:
                after = ' '.join(after_tmp[1:])
                after_sentence = after_tmp[0]
                after_sentence = ' {}'.format(after_sentence if after_sentence.startswith(' ') else after_sentence)

            # Build sentence with answer highlighted
            sentence_answer = '{}{}{} {}{} {}'.format(
                before, before_sentence, HIGHLIGHT_TOKEN, 
                example['answer'], HIGHLIGHT_TOKEN, after_sentence
            ).strip()
            
            # Build paragraph with sentence highlighted
            paragraph_sentence = '{} {} {}'.format(
                before, example['paragraph'][position: position + len(example['answer'])], after
            ).strip()
            
            # Full paragraph with answer highlighted
            paragraph_answer = example['paragraph']

            return {
                'question': example['question'],
                'paragraph_sentence': paragraph_sentence,
                'paragraph_answer': paragraph_answer,
                'sentence_answer': sentence_answer,
                'answer': example['answer'],
                'difficulty': data.get('difficulty', 'basic')
            }
        except Exception as e:
            print(f"  Error processing sample: {str(e)[:50]}")
            return None

    def process_data(self, input_dir, output_dir):
        """Process all JSONL files with detailed reporting"""
        os.makedirs(output_dir, exist_ok=True)
        
        total_stats = {}
        
        for split in ['train', 'validation', 'test']:
            input_file = os.path.join(input_dir, f'{split}.jsonl')
            if not os.path.exists(input_file):
                print(f"  ⊘ {split}: file not found")
                continue
            
            print(f"\n  [{split}]")
            examples = self.jsonline_reader(input_file)
            print(f"    Loaded: {len(examples)} samples")
            
            processed = []
            for data in tqdm(examples, desc=f"  Processing"):
                result = self.process_single_data(data)
                if result is not None:
                    processed.append(result)
            
            # Save processed data
            output_file = os.path.join(output_dir, f'{split}.jsonl')
            with open(output_file, 'w', encoding='utf-8') as f:
                for item in processed:
                    f.write(json.dumps(item, ensure_ascii=False) + '\n')
            
            total_stats[split] = (len(processed), len(examples))
            pct = (len(processed) / len(examples)) * 100 if examples else 0
            print(f"    Processed: {len(processed)}/{len(examples)} ({pct:.0f}%) → {output_file}")
        
        print(f"\n  Summary:")
        total_kept = sum(s[0] for s in total_stats.values())
        total_all = sum(s[1] for s in total_stats.values())
        for split, (kept, total) in total_stats.items():
            print(f"    {split}: {kept}/{total}")
        print(f"    TOTAL: {total_kept}/{total_all} ({(total_kept/total_all)*100:.0f}%)")

print("✓ QGDataProcessor loaded with full architecture")


In [ ]:
# Run data preprocessing directly
processor = QGDataProcessor()

# Verify input directory exists and has files
input_dir = '/content/ViQAG_work/data/examples_ai_flashcard'
output_dir = '/content/ViQAG_work/data/processed_ai_flashcard'

print(f"Input directory: {input_dir}")
if not os.path.exists(input_dir):
    print(f"ERROR: Input directory not found!")
    print(f"Please upload train.jsonl, validation.jsonl, test.jsonl to: {input_dir}")
else:
    files_found = os.listdir(input_dir)
    print(f"Files found: {files_found}")
    
    processor.process_data(
        input_dir=input_dir,
        output_dir=output_dir
    )
    
    print("\nData preprocessing completed!")


In [ ]:
# Verify processed data
import os

processed_dir = '/content/ViQAG_work/data/processed_ai_flashcard'
print("Processed data files:")
for f in os.listdir(processed_dir):
    path = os.path.join(processed_dir, f)
    if os.path.isfile(path) and f.endswith('.jsonl'):
        lines = len(open(path).readlines())
        print(f"  - {f}: {lines} samples")

# Show sample
print("\nSample processed train data:")
with open(os.path.join(processed_dir, 'train.jsonl'), 'r') as f:
    import json
    sample = json.loads(f.readline())
    print(json.dumps(sample, indent=2, ensure_ascii=False))

## Step 5: Setup Training Configuration

In [ ]:
import torch

# Check GPU availability
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Training config
config = {
    'model_name': 'VietAI/vit5-base',
    'batch_size': 16,
    'learning_rate': 3e-4,
    'num_epochs': 5,
    'warmup_steps': 500,
    'max_source_length': 512,
    'max_target_length': 128,
    'gradient_accumulation_steps': 1,
    'seed': 42
}

print("\nTraining Configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")

## Step 6: Fine-tune Answer Extraction (AE) Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import load_dataset
import torch
import json
import os

# Load dataset for Answer Extraction using enhanced format
def load_ae_data(data_dir):
    """Load data for Answer Extraction task - use sentence_answer representation"""
    ae_data = {'train': [], 'validation': [], 'test': []}
    
    for split in ae_data.keys():
        file_path = os.path.join(data_dir, f'{split}.jsonl')
        if not os.path.exists(file_path):
            continue
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    item = json.loads(line.strip())
                    # AE task: sentence_answer (with highlight tokens) → answer
                    ae_data[split].append({
                        'input': item['sentence_answer'],
                        'output': item['answer']
                    })
                except:
                    continue
    return ae_data

# Load data
ae_data = load_ae_data('/content/ViQAG_work/data/processed_ai_flashcard')

print("Answer Extraction (AE) Data:")
print(f"  Train: {len(ae_data['train'])} samples")
print(f"  Validation: {len(ae_data['validation'])} samples")
print(f"  Test: {len(ae_data['test'])} samples")

# Show sample
if ae_data['train']:
    print(f"\nSample AE task:")
    print(f"  Input:  {ae_data['train'][0]['input'][:100]}...")
    print(f"  Output: {ae_data['train'][0]['output']}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset
import torch

# Load fresh model for AE with robust error handling
model_name = 'VietAI/vit5-base'

# Try loading model
try:
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    print("✓ AE Model loaded successfully")
except Exception as e:
    print(f"ERROR loading AE model: {str(e)[:100]}")
    raise

# Try loading tokenizer with fallbacks
try:
    print("Attempting to load AE tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print("✓ AE Tokenizer loaded (AutoTokenizer)")
except Exception as e:
    print(f"AutoTokenizer failed: {str(e)[:100]}")
    try:
        from transformers import T5Tokenizer
        tokenizer = T5Tokenizer.from_pretrained('t5-base')
        print("✓ AE Tokenizer loaded (T5Tokenizer fallback)")
    except Exception as e2:
        print(f"T5Tokenizer also failed: {str(e2)[:100]}")
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained('google/mt5-base', trust_remote_code=False)
        print("✓ AE Tokenizer loaded (mt5-base fallback)")

# Prepare AE dataset
train_dataset = Dataset.from_dict({
    'input': [d['input'] for d in ae_data['train']],
    'output': [d['output'] for d in ae_data['train']]
})

val_dataset = Dataset.from_dict({
    'input': [d['input'] for d in ae_data['validation']],
    'output': [d['output'] for d in ae_data['validation']]
})

def preprocess_function(examples):
    inputs = examples['input']
    outputs = examples['output']
    
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding='max_length')
    
    # Check if tokenizer has as_target_tokenizer method (for compatibility)
    if hasattr(tokenizer, 'as_target_tokenizer'):
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(outputs, max_length=128, truncation=True, padding='max_length')
    else:
        labels = tokenizer(outputs, max_length=128, truncation=True, padding='max_length')
    
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)

# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Training arguments for AE - MEMORY OPTIMIZED
training_args = Seq2SeqTrainingArguments(
    output_dir='/content/ViQAG_work/models/ae_model',
    num_train_epochs=5,
    per_device_train_batch_size=2,  # REDUCED to save memory
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch = 8
    learning_rate=3e-4,
    warmup_steps=500,
    weight_decay=0.01,
    save_total_limit=2,
    logging_steps=50,
    seed=42,
    push_to_hub=False,
    fp16=True,  # Mixed precision training
    dataloader_drop_last=True
)

print("✓ AE Training setup complete (MEMORY OPTIMIZED)")
print("  - Batch size: 2")
print("  - Gradient accumulation: 4 steps")
print("  - Mixed precision (fp16): Enabled")


In [ ]:
# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

In [ ]:
# Train AE model with enhanced monitoring
import torch
import json

print("Starting AE Model Training...")
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

# Train
trainer.train()

print("\n✓ AE Model training completed!")

# Show training results
if hasattr(trainer, 'state') and trainer.state.best_metric:
    print(f"\nTraining Results:")
    print(f"  Best metric: {trainer.state.best_metric:.4f}")
    print(f"  Total steps: {trainer.state.global_step}")

# Save training log for verification
training_log = trainer.state.log_history if hasattr(trainer.state, 'log_history') else []
print(f"\nTraining log entries: {len(training_log)}")
if training_log:
    print(f"  First loss: {training_log[0].get('loss', 'N/A')}")
    print(f"  Last loss: {training_log[-1].get('loss', 'N/A')}")


In [ ]:
# Save AE model
ae_model_dir = '/content/ViQAG_work/models/ae_model_final'
trainer.save_model(ae_model_dir)
tokenizer.save_pretrained(ae_model_dir)
print(f"✓ AE Model saved to {ae_model_dir}")

## Step 7: Fine-tune Question Generation (QG) Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import json
import os

# Load dataset for Question Generation using enhanced format
def load_qg_data(data_dir):
    """Load data for Question Generation task - use paragraph_answer representation"""
    qg_data = {'train': [], 'validation': [], 'test': []}
    
    for split in qg_data.keys():
        file_path = os.path.join(data_dir, f'{split}.jsonl')
        if not os.path.exists(file_path):
            continue
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    item = json.loads(line.strip())
                    # QG task: paragraph_answer (with highlight tokens) → question
                    qg_data[split].append({
                        'input': item['paragraph_answer'],
                        'output': item['question']
                    })
                except:
                    continue
    return qg_data

# Load QG data
qg_data = load_qg_data('/content/ViQAG_work/data/processed_ai_flashcard')

print("Question Generation (QG) Data:")
print(f"  Train: {len(qg_data['train'])} samples")
print(f"  Validation: {len(qg_data['validation'])} samples")
print(f"  Test: {len(qg_data['test'])} samples")

# Show sample
if qg_data['train']:
    print(f"\nSample QG task:")
    print(f"  Input:  {qg_data['train'][0]['input'][:100]}...")
    print(f"  Output: {qg_data['train'][0]['output']}")


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset
import torch
import gc

# Clear GPU memory - remove AE model to free space for QG training
if 'trainer' in locals():
    del trainer
if 'model' in locals():
    del model
if 'tokenizer' in locals():
    del tokenizer

torch.cuda.empty_cache()
gc.collect()
print("✓ GPU memory cleared")

# Load fresh model for QG with robust error handling
model_name = 'VietAI/vit5-base'

# Try loading model
try:
    model_qg = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    print("✓ QG Model loaded successfully")
except Exception as e:
    print(f"ERROR loading QG model: {str(e)[:100]}")
    raise

# Try loading tokenizer with fallbacks
try:
    print("Attempting to load QG tokenizer...")
    tokenizer_qg = AutoTokenizer.from_pretrained(model_name)
    print("✓ QG Tokenizer loaded (AutoTokenizer)")
except Exception as e:
    print(f"AutoTokenizer failed: {str(e)[:100]}")
    try:
        from transformers import T5Tokenizer
        tokenizer_qg = T5Tokenizer.from_pretrained('t5-base')
        print("✓ QG Tokenizer loaded (T5Tokenizer fallback)")
    except Exception as e2:
        print(f"T5Tokenizer also failed: {str(e2)[:100]}")
        from transformers import AutoTokenizer
        tokenizer_qg = AutoTokenizer.from_pretrained('google/mt5-base', trust_remote_code=False)
        print("✓ QG Tokenizer loaded (mt5-base fallback)")

# Prepare QG dataset
train_dataset_qg = Dataset.from_dict({
    'input': [d['input'] for d in qg_data['train']],
    'output': [d['output'] for d in qg_data['train']]
})

val_dataset_qg = Dataset.from_dict({
    'input': [d['input'] for d in qg_data['validation']],
    'output': [d['output'] for d in qg_data['validation']]
})

def preprocess_function_qg(examples):
    inputs = examples['input']
    outputs = examples['output']
    
    model_inputs = tokenizer_qg(inputs, max_length=512, truncation=True, padding='max_length')
    
    # Check if tokenizer has as_target_tokenizer method (for compatibility)
    if hasattr(tokenizer_qg, 'as_target_tokenizer'):
        with tokenizer_qg.as_target_tokenizer():
            labels = tokenizer_qg(outputs, max_length=128, truncation=True, padding='max_length')
    else:
        labels = tokenizer_qg(outputs, max_length=128, truncation=True, padding='max_length')
    
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_dataset_qg = train_dataset_qg.map(preprocess_function_qg, batched=True)
val_dataset_qg = val_dataset_qg.map(preprocess_function_qg, batched=True)

# Data collator
data_collator_qg = DataCollatorForSeq2Seq(tokenizer_qg, model=model_qg)

# Training arguments for QG - MEMORY OPTIMIZED
training_args_qg = Seq2SeqTrainingArguments(
    output_dir='/content/ViQAG_work/models/qg_model',
    num_train_epochs=5,
    per_device_train_batch_size=2,  # REDUCED from 8 to 2
    per_device_eval_batch_size=4,   # REDUCED from 8 to 4
    gradient_accumulation_steps=4,  # NEW: accumulate 4 steps = effective batch 8
    learning_rate=3e-4,
    warmup_steps=500,
    weight_decay=0.01,
    save_total_limit=2,
    logging_steps=50,
    seed=42,
    push_to_hub=False,
    fp16=True,  # NEW: mixed precision training to reduce memory
    dataloader_drop_last=True  # NEW: drop incomplete batches
)

# Trainer for QG
trainer_qg = Seq2SeqTrainer(
    model=model_qg,
    args=training_args_qg,
    train_dataset=train_dataset_qg,
    eval_dataset=val_dataset_qg,
    data_collator=data_collator_qg
)

print("✓ QG Training setup complete (MEMORY OPTIMIZED)")
print("  - Batch size: 2 (was 8)")
print("  - Gradient accumulation: 4 steps")
print("  - Mixed precision (fp16): Enabled")
print("Starting QG model training...")

In [ ]:
# Train QG model
print("Starting QG Model Training...")
print(f"Training samples: {len(train_dataset_qg)}")
print(f"Validation samples: {len(val_dataset_qg)}")

trainer_qg.train()

print("\n✓ QG Model training completed!")

# Show training results
if hasattr(trainer_qg, 'state') and trainer_qg.state.best_metric:
    print(f"\nTraining Results:")
    print(f"  Best metric: {trainer_qg.state.best_metric:.4f}")
    print(f"  Total steps: {trainer_qg.state.global_step}")

# Save training log for verification
training_log_qg = trainer_qg.state.log_history if hasattr(trainer_qg.state, 'log_history') else []
print(f"\nTraining log entries: {len(training_log_qg)}")
if training_log_qg:
    print(f"  First loss: {training_log_qg[0].get('loss', 'N/A')}")
    print(f"  Last loss: {training_log_qg[-1].get('loss', 'N/A')}")


In [ ]:
# Save QG model
qg_model_dir = '/content/ViQAG_work/models/qg_model_final'
trainer_qg.save_model(qg_model_dir)
tokenizer_qg.save_pretrained(qg_model_dir)
print(f"✓ QG Model saved to {qg_model_dir}")

## Step 8: Download Models to Google Drive

In [ ]:
import shutil
import os

# Save to Google Drive
drive_folder = '/content/drive/MyDrive/ViQAG_Models'
os.makedirs(drive_folder, exist_ok=True)

# Copy AE model
ae_dest = os.path.join(drive_folder, 'ae_model_final')
if os.path.exists(ae_dest):
    shutil.rmtree(ae_dest)
shutil.copytree('/content/ViQAG_work/models/ae_model_final', ae_dest)
print(f"✓ AE Model saved to {ae_dest}")

# Copy QG model
qg_dest = os.path.join(drive_folder, 'qg_model_final')
if os.path.exists(qg_dest):
    shutil.rmtree(qg_dest)
shutil.copytree('/content/ViQAG_work/models/qg_model_final', qg_dest)
print(f"✓ QG Model saved to {qg_dest}")

print(f"\n✓ All models saved to Google Drive: {drive_folder}")

## Step 9: Test Pipeline with A Sample

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import gc

# **CRITICAL: Clear GPU memory before loading inference models**
print("Clearing GPU memory...")

# Delete training objects
if 'trainer_qg' in locals():
    del trainer_qg
if 'model_qg' in locals():
    del model_qg
if 'tokenizer_qg' in locals():
    del tokenizer_qg
if 'train_dataset_qg' in locals():
    del train_dataset_qg
if 'val_dataset_qg' in locals():
    del val_dataset_qg

# Force GPU cache clear and garbage collection
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
gc.collect()

print("✓ GPU memory cleared")
print(f"GPU memory available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# ========== Load trained models with robust error handling ==========
ae_model_path = '/content/ViQAG_work/models/ae_model_final'
qg_model_path = '/content/ViQAG_work/models/qg_model_final'

print("\nLoading AE model...")
# Load AE model and tokenizer
try:
    ae_model = AutoModelForSeq2SeqLM.from_pretrained(ae_model_path).to('cuda')
    print("✓ AE Model loaded")
except Exception as e:
    print(f"ERROR loading AE model: {str(e)[:100]}")
    raise

try:
    ae_tokenizer = AutoTokenizer.from_pretrained(ae_model_path)
    print("✓ AE Tokenizer loaded")
except Exception as e:
    print(f"AE Tokenizer failed: {str(e)[:100]}")
    try:
        from transformers import T5Tokenizer
        ae_tokenizer = T5Tokenizer.from_pretrained('t5-base')
        print("✓ AE Tokenizer loaded (T5 fallback)")
    except:
        from transformers import AutoTokenizer
        ae_tokenizer = AutoTokenizer.from_pretrained('google/mt5-base', trust_remote_code=False)
        print("✓ AE Tokenizer loaded (mt5 fallback)")

print("\nLoading QG model...")
# Load QG model and tokenizer
try:
    qg_model = AutoModelForSeq2SeqLM.from_pretrained(qg_model_path).to('cuda')
    print("✓ QG Model loaded")
except Exception as e:
    print(f"ERROR loading QG model: {str(e)[:100]}")
    raise

try:
    qg_tokenizer = AutoTokenizer.from_pretrained(qg_model_path)
    print("✓ QG Tokenizer loaded")
except Exception as e:
    print(f"QG Tokenizer failed: {str(e)[:100]}")
    try:
        from transformers import T5Tokenizer
        qg_tokenizer = T5Tokenizer.from_pretrained('t5-base')
        print("✓ QG Tokenizer loaded (T5 fallback)")
    except:
        from transformers import AutoTokenizer
        qg_tokenizer = AutoTokenizer.from_pretrained('google/mt5-base', trust_remote_code=False)
        print("✓ QG Tokenizer loaded (mt5 fallback)")

print("✓ All models loaded successfully for inference")

In [ ]:
# Test with sample data from validation set
import torch
import numpy as np

print("TESTING MODELS WITH VALIDATION DATA\n")
print("="*70)

# Set models to eval mode and disable gradients
ae_model.eval()
qg_model.eval()

# Load a sample from validation data to test
ae_sample = ae_data['validation'][0] if ae_data['validation'] else None
qg_sample = qg_data['validation'][0] if qg_data['validation'] else None

if ae_sample and qg_sample:
    print("\n✓ Test Sample 1: Answer Extraction (AE)")
    print(f"  Input (with highlight tokens): {ae_sample['input'][:100]}...")
    print(f"  Expected Output: {ae_sample['output']}")
    
    # Step 1: AE - Extract answer from sentence with highlights
    ae_inputs = ae_tokenizer(ae_sample['input'], return_tensors='pt', max_length=512, truncation=True, padding=True).to('cuda')
    
    print(f"  Input tokens shape: {ae_inputs['input_ids'].shape}")
    print(f"  Input token IDs (first 20): {ae_inputs['input_ids'][0, :20]}")
    
    with torch.no_grad():
        # Try generation with better parameters
        ae_outputs = ae_model.generate(
            **ae_inputs, 
            max_length=128,
            min_length=2,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=2,
            length_penalty=2.0,
            temperature=0.7,
            top_p=0.9,
            do_sample=False
        )
    
    print(f"  Output shape: {ae_outputs.shape}")
    print(f"  Output token IDs: {ae_outputs[0]}")
    
    extracted_answer = ae_tokenizer.decode(ae_outputs[0], skip_special_tokens=True)
    print(f"  🤖 Model Output: '{extracted_answer}'")
    
    if not extracted_answer.strip():
        print("  ⚠️ WARNING: Empty output! Checking model predictions...")
        with torch.no_grad():
            logits = ae_model(input_ids=ae_inputs['input_ids'], decoder_input_ids=ae_inputs['input_ids']).logits
            print(f"  Logits shape: {logits.shape}")
            print(f"  Logits mean: {logits.mean():.4f}, std: {logits.std():.4f}")
    
    print()
    
    print("✓ Test Sample 2: Question Generation (QG)")
    print(f"  Input (paragraph): {qg_sample['input'][:100]}...")
    print(f"  Expected Output: {qg_sample['output']}")
    
    # Step 2: QG - Generate question from paragraph
    qg_inputs = qg_tokenizer(qg_sample['input'], return_tensors='pt', max_length=512, truncation=True, padding=True).to('cuda')
    
    print(f"  Input tokens shape: {qg_inputs['input_ids'].shape}")
    
    with torch.no_grad():
        qg_outputs = qg_model.generate(
            **qg_inputs,
            max_length=128,
            min_length=2,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=2,
            length_penalty=2.0,
            temperature=0.7,
            top_p=0.9,
            do_sample=False
        )
    
    print(f"  Output shape: {qg_outputs.shape}")
    
    generated_question = qg_tokenizer.decode(qg_outputs[0], skip_special_tokens=True)
    print(f"  🤖 Model Output: '{generated_question}'")
    
    if not generated_question.strip():
        print("  ⚠️ WARNING: Empty output!")
    
    print("\n" + "="*70)
    print("FULL PIPELINE DEMO")
    print("="*70)
    
    # Full pipeline test
    sample_paragraph = qg_sample['input']
    
    print(f"\nParagraph: {sample_paragraph}\n")
    
    # AE: Extract answer
    ae_inputs = ae_tokenizer(qg_sample['input'], return_tensors='pt', max_length=512, truncation=True, padding=True).to('cuda')
    with torch.no_grad():
        ae_outputs = ae_model.generate(**ae_inputs, max_length=128, min_length=2, num_beams=4, early_stopping=True)
    ae_result = ae_tokenizer.decode(ae_outputs[0], skip_special_tokens=True)
    
    # QG: Generate question
    qg_inputs = qg_tokenizer(qg_sample['input'], return_tensors='pt', max_length=512, truncation=True, padding=True).to('cuda')
    with torch.no_grad():
        qg_outputs = qg_model.generate(**qg_inputs, max_length=128, min_length=2, num_beams=4, early_stopping=True)
    qg_result = qg_tokenizer.decode(qg_outputs[0], skip_special_tokens=True)
    
    print(f"📝 Generated Question: {qg_result if qg_result.strip() else '(empty)'}")
    print(f"✏️ Extracted Answer: {ae_result if ae_result.strip() else '(empty)'}")
    print("="*70)
    
else:
    print("ERROR: No validation samples found!")


In [ ]:
# Quick diagnostic: Test if models are generating at all
import torch

print("MODEL DIAGNOSTIC TEST")
print("="*70)

ae_model.eval()
qg_model.eval()

# Test 1: Simple greedy decoding with AE model
test_input = "Test <hl> answer <hl> here"
print(f"\n1. AE Model Test:")
print(f"   Input: '{test_input}'")

ae_test_inputs = ae_tokenizer(test_input, return_tensors='pt', max_length=64, padding=True).to('cuda')
print(f"   Tokenized input_ids: {ae_test_inputs['input_ids']}")

with torch.no_grad():
    # Try greedy decoding first
    ae_greedy = ae_model.generate(
        input_ids=ae_test_inputs['input_ids'],
        attention_mask=ae_test_inputs['attention_mask'],
        max_length=50,
        num_beams=1,  # Greedy
        do_sample=False
    )

ae_greedy_text = ae_tokenizer.decode(ae_greedy[0], skip_special_tokens=True)
print(f"   Greedy output: '{ae_greedy_text}'")
print(f"   Output tokens: {ae_greedy[0]}")

# Test 2: Check if models have learned anything
print(f"\n2. Model Parameter Check:")
print(f"   AE model params trainable: {sum(p.numel() for p in ae_model.parameters() if p.requires_grad):,}")
print(f"   QG model params trainable: {sum(p.numel() for p in qg_model.parameters() if p.requires_grad):,}")

# Test 3: Check logits
print(f"\n3. AE Model Logits Check:")
with torch.no_grad():
    outputs = ae_model(input_ids=ae_test_inputs['input_ids'], decoder_input_ids=ae_test_inputs['input_ids'])
    print(f"   Logits shape: {outputs.logits.shape}")
    print(f"   Logits range: [{outputs.logits.min():.4f}, {outputs.logits.max():.4f}]")
    print(f"   Top tokens at first position: {torch.topk(outputs.logits[0, 0], 5).indices}")

print("\n" + "="*70)


In [ ]:
# CRITICAL: Check Training Loss to verify if models actually learned
print("\n" + "="*70)
print("TRAINING LOSS VERIFICATION")
print("="*70)

# Get training logs from AE model
ae_training_log = trainer.state.log_history if hasattr(trainer.state, 'log_history') else []
qg_training_log = trainer_qg.state.log_history if hasattr(trainer_qg.state, 'log_history') else []

print("\n📊 AE Model Training Loss:")
if ae_training_log:
    losses = [log.get('loss') for log in ae_training_log if 'loss' in log]
    if losses:
        print(f"  Initial loss: {losses[0]:.4f}")
        print(f"  Final loss: {losses[-1]:.4f}")
        print(f"  Loss reduction: {(losses[0] - losses[-1]):.4f} ({((losses[0] - losses[-1])/losses[0]*100):.1f}%)")
        if losses[-1] > losses[0] * 0.9:
            print("  ⚠️ WARNING: Loss barely decreased! Model may not have trained properly.")
        else:
            print("  ✓ Loss decreased significantly - Training worked!")
else:
    print("  ⚠️ No training log found for AE model")

print("\n📊 QG Model Training Loss:")
if qg_training_log:
    losses_qg = [log.get('loss') for log in qg_training_log if 'loss' in log]
    if losses_qg:
        print(f"  Initial loss: {losses_qg[0]:.4f}")
        print(f"  Final loss: {losses_qg[-1]:.4f}")
        print(f"  Loss reduction: {(losses_qg[0] - losses_qg[-1]):.4f} ({((losses_qg[0] - losses_qg[-1])/losses_qg[0]*100):.1f}%)")
        if losses_qg[-1] > losses_qg[0] * 0.9:
            print("  ⚠️ WARNING: Loss barely decreased! Model may not have trained properly.")
        else:
            print("  ✓ Loss decreased significantly - Training worked!")
else:
    print("  ⚠️ No training log found for QG model")

print("\n" + "="*70)
print("RECOMMENDATION:")
if (ae_training_log and any('loss' in log for log in ae_training_log)) or \
   (qg_training_log and any('loss' in log for log in qg_training_log)):
    print("✓ Models appear to have trained.")
    print("  If outputs are still garbled, try retraining with:")
    print("  - More epochs (10-15)")
    print("  - Lower learning rate (1e-4)")
    print("  - Larger batch size if GPU memory permits")
else:
    print("⚠️ No training logs found. Retrain the models!")
print("="*70)


## Summary

✓ **Completed:**
- Vietnamese text tokenization with Underthesea
- Rich representations with highlight tokens (`<hl>`)
  - `sentence_answer`: Sentence-level representation for Answer Extraction
  - `paragraph_answer`: Paragraph-level representation for Question Generation
  - `paragraph_sentence`: Highlighted key information
- Answer Extraction (AE) model fine-tuned with sentence context
- Question Generation (QG) model fine-tuned with full paragraph context
- Both models saved to Google Drive
- Pipeline tested with sample paragraph

**Key Features:**
- Robust error handling for malformed JSON
- Flexible processing (skips samples where answer not found)
- Underthesea for Vietnamese-aware sentence tokenization
- Highlight token mechanism helps models focus on target answer

**Next Steps:**
1. Download models from Google Drive
2. Integrate with your ViQAG repo
3. Use in production for flashcard generation
4. Fine-tune with more Vietnamese AI domain data as needed
